# S100B Binder Design — Boltz 2.1 API
## `protein:design` → generate candidates + `structure-and-binding` → deep validate top hits

Tests 3 candidate binding sites. Boltz generates novel sequences targeting each site; no candidate list needed.

| Site | Residues (1-based PDB) | Notes |
|------|------------------------|-------|
| `helix_1` | 32–39 | Outer α-helix, safe from Ca/Zn sites |
| `helix_2` | 48–59 | Outer α-helix, safe from Ca/Zn sites |
| `hydrophobic_cleft` | 52,56,59,60,69,76,80,83,84 | Ca²⁺-dependent cleft, literature-validated |

**Avoided** (Ca/Zn binding + endogenous): 16–19, 26, 62–73, 86–91, Phe88/Phe89/Trp90.

> Residue indices in the API payload are **0-based** (PDB number − 1).

Ref: https://pmc.ncbi.nlm.nih.gov/articles/PMC6405260/

In [40]:
import subprocess, json, os, re, urllib.request
import pandas as pd
from pathlib import Path
import yaml

BOLTZ_API  = Path.home() / ".local/bin/boltz-api"
ROOT_DIR   = Path("/Users/bridget/Desktop/projects/igemmodel/boltz-experiments").resolve()
PDB_PATH   = Path("s100b_binder/s100b_dimer.pdb").resolve()
PAYLOAD_DIR = ROOT_DIR / "payloads"
MODEL      = "boltz-2.1"
DATE       = "20260627"       # bump to force a fresh batch
NUM_PROTEINS = 200            # per site; scale to 1000+ for a real campaign

ROOT_DIR.mkdir(exist_ok=True)
PAYLOAD_DIR.mkdir(exist_ok=True)

# Load API key from .env (avoids OAuth token expiry issues)
_env = dict(l.strip().split(" = ", 1) for l in open(".env") if " = " in l)
BOLTZ_ENV = {**os.environ, "BOLTZ_API_KEY": list(_env.values())[0].strip()}

_ANSI = re.compile(r'\x1b\[[0-9;]*m')

def boltz(*args, input_data=None):
    """Run boltz-api CLI, strip ANSI codes, return parsed JSON."""
    r = subprocess.run(
        [str(BOLTZ_API)] + list(args),
        capture_output=True, text=True, env=BOLTZ_ENV,
        input=input_data,
    )
    raw = (_ANSI.sub('', r.stdout) or _ANSI.sub('', r.stderr)).strip()
    if r.returncode != 0 and not raw:
        raise RuntimeError(f"boltz-api error:\n{r.stderr}")
    return raw  # caller parses if needed

print("boltz-api:", boltz("--version"))

boltz-api: boltz-api version 0.35.0


## 1. Download S100B target structure (3D0Y)

In [41]:
os.makedirs("s100b_binder", exist_ok=True)

urllib.request.urlretrieve(
    "https://files.rcsb.org/download/3D0Y.pdb",
    "s100b_binder/3D0Y.pdb"
)

clean = []
with open("s100b_binder/3D0Y.pdb") as f:
    for line in f:
        if line.startswith("ATOM") and line[21] in ("A", "B"):
            clean.append(line)

with open("s100b_binder/s100b_dimer.pdb", "w") as f:
    f.writelines(clean)
    f.write("END\n")

print(f"{len(clean)} ATOM lines, chains: {sorted(set(l[21] for l in clean))}")
print("First:", clean[0][17:26].strip(), " Last:", clean[-1][17:26].strip())

1471 ATOM lines, chains: ['A', 'B']
First: SER A   1  Last: GLU B  89


## 2. Extract S100B sequence (chain A)

In [42]:
AA3 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V",
}

seen = set()
seq = []
with open("s100b_binder/3D0Y.pdb") as f:
    for line in f:
        if line.startswith("ATOM") and line[21] == "A":
            res_id = int(line[22:26])
            res_name = line[17:20].strip()
            if res_id not in seen:
                seen.add(res_id)
                seq.append(AA3.get(res_name, "X"))

S100B_SEQ = "".join(seq)
print(f"S100B chain A: {len(S100B_SEQ)} residues")
print(S100B_SEQ)

S100B chain A: 89 residues
SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFE


## 3. Define binding sites & candidates

Add your candidate sequences to the `candidates` list for each site.  
Each entry is `(name, sequence)`.  
**TRTK-12** is a literature-validated S100B binder included as a positive control.

In [43]:
# Residue indices are 0-based in the Boltz API (PDB number − 1).

# Shared avoidance zone: Ca/Zn binding + endogenous binding residues (0-based)
_BASE_NON_BINDING = (
    list(range(15, 19))  # PDB 16–19
    + [25]               # PDB 26
    + list(range(61, 73))# PDB 62–73
    + list(range(85, 89))# PDB 86–89 (chain A only has 89 residues → max index 88)
)

def _non_binding(epitope):
    """Remove any epitope overlap so the constraint doesn't contradict itself."""
    return sorted(set(_BASE_NON_BINDING) - set(epitope))


SITES = {
    "helix_1": {
        "description": "Outer α-helix (PDB 32–39), safe from Ca/Zn sites",
        "epitope":     list(range(31, 39)),  # PDB 32–39 → 0-based 31–38
    },
    "helix_2": {
        "description": "Outer α-helix (PDB 48–59), safe from Ca/Zn sites",
        "epitope":     list(range(47, 59)),  # PDB 48–59 → 0-based 47–58
    },
    "hydrophobic_cleft": {
        "description": "Ca²⁺-dependent hydrophobic cleft (literature-validated)",
        # PDB 52,56,59,60,69,76,80,83,84 → subtract 1
        "epitope":     [51, 55, 58, 59, 68, 75, 79, 82, 83],
    },
}

for site, cfg in SITES.items():
    cfg["non_binding"] = _non_binding(cfg["epitope"])
    print(f"{site}:")
    print(f"  epitope (0-based):      {cfg['epitope']}")
    print(f"  non_binding (0-based):  {cfg['non_binding']}")

helix_1:
  epitope (0-based):      [31, 32, 33, 34, 35, 36, 37, 38]
  non_binding (0-based):  [15, 16, 17, 18, 25, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 85, 86, 87, 88]
helix_2:
  epitope (0-based):      [47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]
  non_binding (0-based):  [15, 16, 17, 18, 25, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 85, 86, 87, 88]
hydrophobic_cleft:
  epitope (0-based):      [51, 55, 58, 59, 68, 75, 79, 82, 83]
  non_binding (0-based):  [15, 16, 17, 18, 25, 61, 62, 63, 64, 65, 66, 67, 69, 70, 71, 72, 85, 86, 87, 88]


## 4. Visualize target with binding sites highlighted

In [44]:
import py3Dmol

with open("s100b_binder/s100b_dimer.pdb") as f:
    pdb_str = f.read()

SITE_COLORS = {
    "helix_1":           "yellow",
    "helix_2":           "orange",
    "hydrophobic_cleft": "red",
}
AVOID_RESIDUES = list(range(16, 20)) + [26] + list(range(62, 74)) + list(range(86, 92))

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_str, "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "lightblue"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "lightgrey"}})

for site, cfg in SITES.items():
    color = SITE_COLORS[site]
    for res_0based in cfg["epitope"]:
        res_pdb = res_0based + 1   # convert back to 1-based for py3Dmol/PDB
        view.setStyle({"chain": "A", "resi": res_pdb},
                      {"cartoon": {"color": color}, "stick": {"color": color}})

for res in AVOID_RESIDUES:
    view.setStyle({"chain": "A", "resi": res},
                  {"cartoon": {"color": "grey"}, "stick": {"color": "grey", "radius": 0.15}})

view.addLabel("helix_1 (yellow)  helix_2 (orange)  cleft (red)  avoid (grey)",
              {"fontSize": 12, "backgroundOpacity": 0.6, "alignment": "topCenter"})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 5. Build `protein:design` payloads and estimate cost

Each YAML file is written to `boltz-experiments/payloads/`. The `@data://` prefix tells the CLI to embed the PDB file as base64 at submission time.

In [45]:
import base64

# Download CIF (required — API only accepts chemical/x-cif, not PDB)
CIF_PATH = Path("s100b_binder/3D0Y.cif")
if not CIF_PATH.exists():
    urllib.request.urlretrieve("https://files.rcsb.org/download/3D0Y.cif", CIF_PATH)
    print(f"Downloaded {CIF_PATH}")

# Embed as base64 directly — @data:// directives aren't expanded inside @yaml:// files
CIF_B64 = base64.b64encode(CIF_PATH.read_bytes()).decode()


def build_payload(site, cfg, num_proteins=NUM_PROTEINS):
    return {
        "num_proteins": num_proteins,
        "target": {
            "type": "structure_template",
            "structure": {
                "type": "base64",
                "media_type": "chemical/x-cif",
                "data": CIF_B64,
            },
            "chain_selection": {
                "A": {
                    "chain_type": "polymer",
                    "crop_residues": "all",
                    "epitope_residues":     cfg["epitope"],
                    "non_binding_residues": cfg["non_binding"],
                },
                "B": {
                    "chain_type": "polymer",
                    "crop_residues": "all",
                },
            },
        },
        "binder_specification": {
            "type": "no_template",
            "modality": "peptide",
            "entities": [
                {"type": "designed_protein", "chain_ids": ["C"], "value": "10..20"},
            ],
        },
    }


payload_paths = {}
for site, cfg in SITES.items():
    p = PAYLOAD_DIR / f"design_{site}.yaml"
    with open(p, "w") as f:
        yaml.dump(build_payload(site, cfg), f, default_flow_style=False, allow_unicode=True)
    payload_paths[site] = p
    print(f"Wrote {p}")

print()

# Estimate cost for all sites
cost_rows = []
for site, path in payload_paths.items():
    raw = boltz("protein:design", "estimate-cost", "--input", f"@yaml://{path}")
    est = json.loads(raw)
    cost_rows.append({
        "site":              site,
        "num_proteins":      NUM_PROTEINS,
        "cost_per_design_usd": float((est.get("breakdown") or {}).get("cost_per_unit_usd") or 0),
        "total_usd":         float(est.get("estimated_cost_usd") or 0),
    })

cost_df = pd.DataFrame(cost_rows)
print(f"Grand total: ${cost_df['total_usd'].sum():.4f} for {cost_df['num_proteins'].sum()} designs")
cost_df

Downloaded s100b_binder/3D0Y.cif
Wrote /Users/bridget/Desktop/projects/igemmodel/boltz-experiments/payloads/design_helix_1.yaml
Wrote /Users/bridget/Desktop/projects/igemmodel/boltz-experiments/payloads/design_helix_2.yaml
Wrote /Users/bridget/Desktop/projects/igemmodel/boltz-experiments/payloads/design_hydrophobic_cleft.yaml

Grand total: $15.0000 for 600 designs


,site,num_proteins,cost_per_design_usd,total_usd
0,helix_1,200,0.025,5.0
1,helix_2,200,0.025,5.0
2,hydrophobic_cleft,200,0.025,5.0


## 6. Submit design jobs

Run `estimate-cost` above first and confirm the total before running this cell.  
Idempotency key = `s100b-{site}-{DATE}` — re-submitting returns the same job ID without re-billing.

In [46]:
job_ids = {}
for site, path in payload_paths.items():
    run_name = f"s100b-design-{site}-{DATE}"
    raw = boltz(
        "protein:design", "start",
        "--idempotency-key", run_name,
        "--input", f"@yaml://{path}",
        "--raw-output", "--transform", "id",
    )
    job_id = raw.strip()
    job_ids[site] = job_id
    print(f"{site}: job {job_id}")

print("\nJob IDs saved. Run the next cell to download results.")

helix_1: job prot_des_zjYf4LTVYSF3Nu6l43BN
helix_2: job prot_des_fkd0M1UAqJi59DgOLp8g
hydrophobic_cleft: job prot_des_vpxExBbOmikQcaDVLaaS

Job IDs saved. Run the next cell to download results.


## 7. Download results and rank

`download-results` polls until done (200 designs takes a few minutes).  
Results land in `boltz-experiments/{run-name}/results/index.jsonl`, one design per line.

**Primary ranking metric**: `binding_confidence` (higher = better).  
Tiebreakers: `iptm` (higher) and `min_interaction_pae` (lower).

In [48]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def download_site(site, job_id):
    run_name = f"s100b-design-{site}-{DATE}"
    boltz(
        "download-results",
        "--id", job_id,
        "--name", run_name,
        "--root-dir", str(ROOT_DIR),
        "--poll-interval-seconds", "30",
    )
    return site, ROOT_DIR / run_name

run_dirs = {}
with ThreadPoolExecutor(max_workers=len(job_ids)) as ex:
    futures = {ex.submit(download_site, site, jid): site for site, jid in job_ids.items()}
    for fut in as_completed(futures):
        site, run_dir = fut.result()
        run_dirs[site] = run_dir
        print(f"✓ {site} → {run_dir}")

print("\nAll downloads complete.")

✓ helix_2 → /Users/bridget/Desktop/projects/igemmodel/boltz-experiments/s100b-design-helix_2-20260627
✓ helix_1 → /Users/bridget/Desktop/projects/igemmodel/boltz-experiments/s100b-design-helix_1-20260627
✓ hydrophobic_cleft → /Users/bridget/Desktop/projects/igemmodel/boltz-experiments/s100b-design-hydrophobic_cleft-20260627

All downloads complete.


## 8. Rank designs across all sites

In [52]:
import jsonlines

records = []
for site, run_dir in run_dirs.items():
    index_path = run_dir / "results/index.jsonl"
    if not index_path.exists():
        print(f"Missing index: {index_path}")
        continue
    with jsonlines.open(index_path) as reader:
        for item in reader:
            m = item.get("metrics", {})
            binder_seq = next(
                (e["value"] for e in item.get("entities", []) if "C" in e.get("chain_ids", [])),
                ""
            )
            rel_cif = item.get("paths", {}).get("structure", "")
            records.append({
                "site":                site,
                "id":                  item["id"],
                "sequence":            binder_seq,
                "binding_confidence":  m.get("binding_confidence"),
                "iptm":                m.get("iptm"),
                "min_interaction_pae": m.get("min_interaction_pae"),
                "structure_confidence":m.get("structure_confidence"),
                "cif_path":            str(run_dir / rel_cif) if rel_cif else "",
            })

results_df = (
    pd.DataFrame(records)
    .sort_values("binding_confidence", ascending=False)
    .reset_index(drop=True)
)
print(f"{len(results_df)} designs total")
results_df.groupby("site")[["binding_confidence","iptm","min_interaction_pae"]].describe()

600 designs total


binding_confidence                                    \
                               count      mean       std           min   
site                                                                     
helix_1                        200.0  0.001653  0.004423  4.449618e-07   
helix_2                        200.0  0.000755  0.001512  2.626492e-06   
hydrophobic_cleft              200.0  0.001662  0.003115  2.761917e-06   

                                                            iptm            \
                        25%       50%       75%       max  count      mean   
site                                                                         
helix_1            0.000009  0.000223  0.001525  0.049024  200.0  0.793418   
helix_2            0.000027  0.000101  0.000937  0.014346  200.0  0.785193   
hydrophobic_cleft  0.000124  0.000631  0.001750  0.027066  200.0  0.793245   

                   ...                     min_interaction_pae            \
                   ...       75%       max               count      mean   
site               ...                                                     
helix_1            ...  0.848809  0.946956               200.0  2.416239   
helix_2            ...  0.837125  0.950069               200.0  2.419662   
hydrophobic_cleft  ...  0.836952  0.948893               200.0  2.340152   

                                                                               
                        std       min       25%       50%       75%       max  
site                                                                           
helix_1            1.039869  0.880599  1.798387  2.215190  2.835225  9.183454  
helix_2            1.010469  0.756069  1.807973  2.183778  2.799679  8.410995  
hydrophobic_cleft  0.678322  0.793872  1.903709  2.234127  2.690058  4.933686  

[3 rows x 24 columns]

## 9. Visualize top hits per site

In [53]:
import py3Dmol

TOP_N = 3  # top designs per site to show

panels = []
for site in SITES:
    top = results_df[results_df["site"] == site].head(TOP_N)
    for _, row in top.iterrows():
        cif_path = Path(row["cif_path"])
        if cif_path.exists():
            label = f"{site}\n{row['id']}\nbind={row['binding_confidence']:.3f} ipTM={row['iptm']:.3f}"
            panels.append((label, cif_path.read_text()))

ncols = min(len(panels), 3) if panels else 1
nrows = max((len(panels) + ncols - 1) // ncols, 1)
view  = py3Dmol.view(width=420 * ncols, height=360 * nrows, viewergrid=(nrows, ncols))

for i, (label, cif_text) in enumerate(panels):
    r, c = divmod(i, ncols)
    view.addModel(cif_text, "cif", viewer=(r, c))
    view.setBackgroundColor("white", viewer=(r, c))
    view.setStyle({}, {"cartoon": {"color": "lightgrey"}}, viewer=(r, c))
    view.setStyle({"chain": "C"}, {"cartoon": {"color": "magenta"},
                                    "stick":   {"color": "magenta", "radius": 0.2}},
                  viewer=(r, c))
    view.addLabel(label, {"fontSize": 10, "backgroundOpacity": 0.6,
                          "alignment": "topCenter"}, viewer=(r, c))
    view.zoomTo(viewer=(r, c))

view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.